In [1]:
# CELL 1: SYSTEM DEPENDENCIES & DYNAMIC DIRECTORY CONFIGURATION
# =========================================================================================
# Pipeline Stage: System Setup
# Algorithm / Toolkit: Python Standard OS Framework, PyTorch Core, MediaPipe Vision Engine
# =========================================================================================

import os
import cv2
import pandas as pd
import numpy as np
import scipy.signal as signal
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Verify hardware acceleration state (Supports CUDA, Intel XPU, Apple Silicon MPS, or CPU)
if hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch, "backends") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# System Filepath Configurations
# System Filepath Configurations (Enforced clean OS normalization)
# System Filepath Configurations (Pointed to the correct active directory structure)
BASE_PROJECT_DIR = os.path.normpath("D:/HACKATHON11")
BASE_DATASET_DIR = os.path.normpath(os.path.join(BASE_PROJECT_DIR, "vitalscan-clinic/mcd_rppg_600_patients"))
CSV_FILE_PATH    = os.path.normpath(os.path.join(BASE_DATASET_DIR, "db.csv"))
OUTPUT_SIGNAL_DIR = os.path.normpath(os.path.join(BASE_DATASET_DIR, "approach3_extracted_signals"))

# Enforce clean workspace environment
os.makedirs(OUTPUT_SIGNAL_DIR, exist_ok=True)

print(f"✅ Cell 1 Complete: Dependencies verified. Compute Target initialized to: {DEVICE}")

✅ Cell 1 Complete: Dependencies verified. Compute Target initialized to: cuda


In [2]:
# CELL 2: VISUAL STABILIZATION & DYNAMIC 8-ROI SKIN SEGMENTATION ENGINE (UPDATED TASKS API)
# =========================================================================================
# Pipeline Stage: Face Mesh Alignment, Motion Correction, and Multi-Zone Skin Extraction
# Algorithms: MediaPipe Tasks FaceLandmarker, Affine Transformation Matrix Warping, 
#             YCbCr Color Space Transformation, Otsu's Automatic Thresholding
# =========================================================================================

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Define explicit topological MediaPipe index landmarks for 8 physiological zones
ROI_LANDMARKS = {
    "left_forehead":   [68, 107, 66, 105, 70],
    "mid_forehead":    [109, 67, 103, 54, 21],
    "right_forehead":  [298, 336, 296, 334, 300],
    "left_cheek":      [118, 119, 100, 142, 123, 50, 205],
    "right_cheek":     [347, 348, 329, 371, 352, 280, 425],
    "nose_bridge":     [6, 197, 195, 5],
    "left_pararenal":  [116, 117, 47, 101, 111],
    "right_pararenal": [345, 346, 277, 330, 340]
}

# Standardized anchoring coordinates used to calculate the stabilizing Affine Warping Matrix.
# Aligns every frame to a rigid, front-facing model plane to eliminate motion artifacts.
STABILIZED_ANCHORS = np.array([
    [30, 20],   # Left Eye Outer Corner Anchor
    [98, 20],   # Right Eye Outer Corner Anchor
    [64, 80]    # Nose Tip Anchor
], dtype=np.float32)

def compute_affine_stabilization(landmarks, w, h):
    """
    Calculates the 2D Affine Transformation matrix based on three reliable tracking landmarks:
    Left Eye Corner (Idx 33), Right Eye Corner (Idx 263), and Nose Tip (Idx 1).
    """
    p1 = np.array([landmarks[33].x * w, landmarks[33].y * h], dtype=np.float32)
    p2 = np.array([landmarks[263].x * w, landmarks[263].y * h], dtype=np.float32)
    p3 = np.array([landmarks[1].x * w, landmarks[1].y * h], dtype=np.float32)
    
    current_src_pts = np.stack([p1, p2, p3])
    affine_matrix = cv2.getAffineTransform(current_src_pts, STABILIZED_ANCHORS)
    return affine_matrix

def segment_skin_otsu(roi_bgr_image):
    """
    Transforms the frame slice into YCbCr space and runs Otsu's Thresholding 
    ONLY on non-zero pixels to discard non-skin pixels without background bias.
    """
    if roi_bgr_image.size == 0:
        return None
    
    ycbcr = cv2.cvtColor(roi_bgr_image, cv2.COLOR_BGR2YCrCb)
    _, _, cb = cv2.split(ycbcr)
    
    # Isolate non-black pixels (where the actual ROI patch lives)
    non_zero_mask = (roi_bgr_image[..., 0] > 0) & (roi_bgr_image[..., 1] > 0) & (roi_bgr_image[..., 2] > 0)
    
    if not np.any(non_zero_mask):
        return None
        
    # Extract only valid patch pixels to build a clean histogram free of background bias
    valid_pixels = cb[non_zero_mask]
    
    try:
        _, binary_skin_mask_flat = cv2.threshold(valid_pixels, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Reconstruct the 2D spatial mask
        binary_skin_mask = np.zeros_like(cb)
        binary_skin_mask[non_zero_mask] = binary_skin_mask_flat.flatten()
        return binary_skin_mask
    except Exception:
        # Fallback if thresholding fails due to uniform color profile
        return non_zero_mask.astype(np.uint8) * 255

# --- INITIALIZE MEDIAPIPE VISION TASK CONFIGURATION ---
MODEL_PATH = "D:/HACKATHON11/face_landmarker.task"
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)

print("🚀 Creating Detector instance from vision task options...")
detector = vision.FaceLandmarker.create_from_options(options)

print("✅ Cell 2 Complete: Tasks API successfully bound. Stabilization and segmentation ready.")

🚀 Creating Detector instance from vision task options...
✅ Cell 2 Complete: Tasks API successfully bound. Stabilization and segmentation ready.


In [3]:
# CELL 3: KLT FEATURE TRACKING & MATHEMATICAL POS SIGNAL EXTRACTION ENGINE
# =========================================================================================
# Pipeline Stage: Temporal Point Tracking & Chrominance-Invariant Signal Processing
# Algorithms: Kanade-Lucas-Tomasi (KLT) Optical Flow, Plane-Orthogonal-to-Skin (POS) rPPG
# =========================================================================================

# KLT Tracking configuration parameters
LKT_PARAMS = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

def track_roi_features_klt(prev_gray, curr_gray, prev_points):
    """
    Applies the Kanade-Lucas-Tomasi (KLT) sparse optical flow algorithm to update
    the localized point arrays from the previous frame to the current frame.
    """
    if prev_points is None or len(prev_points) == 0:
        return None, None
        
    next_points, status, error = cv2.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, prev_points, None, **LKT_PARAMS
    )
    
    # Filter out points that lost tracking stability
    valid_idx = (status == 1).reshape(-1)
    if not np.any(valid_idx):
        return None, None
        
    return next_points[valid_idx], prev_points[valid_idx]

def apply_pos_rppg(rgb_time_series):
    """
    Core Mathematical Execution of Plane-Orthogonal-to-Skin (POS) Algorithm.
    Projects raw RGB traces onto a plane orthogonal to the skin tone vector 
    to robustly extract specular-free blood volume pulse (BVP) variations.
    """
    # rgb_time_series shape: [N, 3] where columns represent R, G, B channels
    C = np.array(rgb_time_series, dtype=np.float32).T  # Transpose to [3, N]
    
    # Step 1: Compute rolling mean using a standard windowing approach (or global mean if short)
    # For simplicity and standard block processing, we use a global zero-phase normalization block
    mean_C = np.mean(C, axis=1, keepdims=True) + 1e-6
    C_norm = C / mean_C
    
    # Step 2: Define projection matrix matrix values orthogonal to skin tones
    # P matrix definitions for the two projection orthogonal vectors
    S1 = 3.0 * C_norm[0, :] - 2.0 * C_norm[1, :]
    S2 = 1.5 * C_norm[0, :] + 1.0 * C_norm[1, :] - 1.5 * C_norm[2, :]
    
    # Step 3: Compute final BVP signal via standard tuning projections
    std_S1 = np.std(S1)
    std_S2 = np.std(S2) + 1e-6
    
    alpha = std_S1 / std_S2
    h_bvp = S1 - (alpha * S2)
    
    return h_bvp

print("✅ Cell 3 Complete: KLT tracking layers and POS matrix math pipelines successfully initialized.")

✅ Cell 3 Complete: KLT tracking layers and POS matrix math pipelines successfully initialized.


In [ ]:
# CELL 4: THREAD-STABILIZED PREPROCESSING & FILTERING PIPELINE WITH SQA
# =========================================================================================
# Pipeline Stage: Safe Parallel Video Parsing, Signal Transformation, SQA, and Storage
# Algorithms: ThreadPoolExecutor Multi-Threading, 6th-Order Butterworth Filter, 
#              Signal Quality Assessment (SQA) Signal-to-Noise Gating, Matrix Serialization
# =========================================================================================

import os
import cv2
import numpy as np
import pandas as pd
# from tqdm.notebook import tqdm
from tqdm import tqdm
import scipy.signal as signal
from concurrent.futures import ThreadPoolExecutor, as_completed
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def butter_bandpass_filter(data, lowcut=0.7, highcut=4.0, fps=30.0, order=6):
    """Applies a high-order zero-phase forward-backward Butterworth filter."""
    nyq = 0.5 * fps
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, data)

def assess_signal_quality(bvp_signal, fps=30.0):
    """Signal Quality Assessment (SQA) Module measuring Skewness and SNR."""
    n = len(bvp_signal)
    if n < 3:
        return 0.0, 0.0
    mean = np.mean(bvp_signal)
    std = np.std(bvp_signal) + 1e-6
    skewness = (sum((bvp_signal - mean) ** 3) / n) / (std ** 3)
    
    fft_vals = np.abs(np.fft.rfft(bvp_signal))
    freqs = np.fft.rfftfreq(n, d=1.0/fps)
    
    band_mask = (freqs >= 0.7) & (freqs <= 4.0)
    in_band_power = np.sum(fft_vals[band_mask] ** 2)
    total_power = np.sum(fft_vals ** 2) + 1e-6
    snr = in_band_power / total_power
    
    is_clean = (abs(skewness) < 2.0) and (snr > 0.4)
    return float(snr), 1.0 if is_clean else 0.0

def process_single_video_thread(task_info):
    """
    Worker function executed by individual threads.
    Thread-safe processing using a newly generated pipeline context per file.
    """
    video_input_path, output_npy_path, model_path, roi_landmarks_dict = task_info
    
    if os.path.exists(output_npy_path):
        return True # Skip pre-computed runs
        
    # Initialize an isolated MediaPipe instance for this thread context
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options, running_mode=vision.RunningMode.IMAGE, num_faces=1
    )
    local_detector = vision.FaceLandmarker.create_from_options(options)
    
    cap = cv2.VideoCapture(video_input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps): 
        fps = 30.0
        
    roi_traces = {zone: [] for zone in roi_landmarks_dict.keys()}
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
            
        h, w, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        
        detection_result = local_detector.detect(mp_image)
        
        if detection_result.face_landmarks:
            landmarks = detection_result.face_landmarks[0]
            affine_mat = compute_affine_stabilization(landmarks, w, h)
            warped_frame = cv2.warpAffine(frame, affine_mat, (128, 128))
            
            for zone_name, indices in roi_landmarks_dict.items():
                mask = np.zeros((h, w), dtype=np.uint8)
                pts = np.array([(int(landmarks[idx].x * w), int(landmarks[idx].y * h)) for idx in indices])
                cv2.fillConvexPoly(mask, pts, 255)
                
                roi_crop = cv2.bitwise_and(frame, frame, mask=mask)
                skin_mask = segment_skin_otsu(roi_crop)
                
                if skin_mask is not None:
                    final_mask = cv2.bitwise_and(mask, skin_mask)
                    mean_color = cv2.mean(frame_rgb, mask=final_mask)[:3]
                else:
                    mean_color = cv2.mean(frame_rgb, mask=mask)[:3]
                    
                roi_traces[zone_name].append(mean_color)
        else:
            for zone_name in roi_landmarks_dict.keys():
                if len(roi_traces[zone_name]) > 0:
                    roi_traces[zone_name].append(roi_traces[zone_name][-1])
                else:
                    roi_traces[zone_name].append([0.0, 0.0, 0.0])
                    
    cap.release()
    local_detector.close()
    
    processed_roi_signals = []
    quality_metrics = []
    
    for zone_name in roi_landmarks_dict.keys():
        trace_arr = np.array(roi_traces[zone_name])
        if len(trace_arr) < 10:
            trace_arr = np.zeros((600, 3))
            
        pos_signal = apply_pos_rppg(trace_arr)
        filtered_signal = butter_bandpass_filter(pos_signal, fps=fps)
        snr_val, clean_flag = assess_signal_quality(filtered_signal, fps=fps)
        
        processed_roi_signals.append(filtered_signal)
        quality_metrics.append([snr_val, clean_flag])
        
    payload_signals = np.stack(processed_roi_signals)
    payload_metrics = np.array(quality_metrics)
    
    storage_dict = {"signals": payload_signals, "metrics": payload_metrics}
    np.save(output_npy_path, storage_dict, allow_pickle=True)
    return True

# --- EXECUTABLE SCHEDULER (PATH-STABILIZED FOR WINDOWS) ---
master_df = pd.read_csv(CSV_FILE_PATH)

# FIXED: Removed dynamic hardcoded indexing splits to scale across the whole cohort automatically
unique_pids = master_df['patient_id'].unique()
development_df = master_df.reset_index(drop=True)

tasks = []
for index, row in development_df.iterrows():
    video_filename = str(row['video']).replace('\\', '/').split('/')[-1]
    video_input_path = os.path.normpath(os.path.join(BASE_DATASET_DIR, 'video', video_filename))
    
    output_npy_basename = f"{os.path.splitext(video_filename)[0]}.npy"
    output_npy_path = os.path.normpath(os.path.join(OUTPUT_SIGNAL_DIR, output_npy_basename))
    
    if os.path.exists(video_input_path):
        tasks.append((video_input_path, output_npy_path, MODEL_PATH, ROI_LANDMARKS))

# FIXED: Dynamic worker counts utilizing maximum allowable processor capabilities
max_workers = max(1, os.cpu_count() - 2) 
print(f"🎬 Initializing path-stabilized ThreadPool for {len(tasks)} videos using {max_workers} threads...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_single_video_thread, task): task for task in tasks}
    
    with tqdm(total=len(futures), desc="⚡ Threaded Extraction") as pbar:
        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception as exc:
                print(f"⚠️ Worker thread raised an internal exception: {exc}")
            pbar.update(1)

print("🎉 Cell 4 Extractor Stable: Data successfully cached via thread pools.")

🎬 Initializing path-stabilized ThreadPool for 3600 videos using 62 threads...


⚡ Threaded Extraction:   0%|          | 0/3600 [00:00<?, ?it/s]

In [4]:
# CELL 4: THREAD-STABILIZED PREPROCESSING & FILTERING PIPELINE WITH SQA (GPU ACCELERATED)
# =========================================================================================
# Pipeline Stage: Parallel Video Parsing, GPU Signal Transformation, SQA, and Storage
# Algorithms: ThreadPoolExecutor Multi-Threading, MediaPipe GPU Delegate Inference,
#             Zero-Phase Butterworth Filter, SQA SNR Metric Gating
# =========================================================================================

import os
import cv2
import numpy as np
import pandas as pd
from tqdm.std import tqdm as text_tqdm
import scipy.signal as signal
from concurrent.futures import ThreadPoolExecutor, as_completed
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

def butter_bandpass_filter(data, lowcut=0.7, highcut=4.0, fps=15.0, order=6):
    """Applies a high-order zero-phase forward-backward Butterworth filter with boundary padding."""
    nyq = 0.5 * fps
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    # CRITICAL FIX: Add safe data symmetric reflection padding to mitigate transient filter edge artifacts
    return signal.filtfilt(b, a, data, padtype='even')

def assess_signal_quality(bvp_signal, fps=15.0):
    """Signal Quality Assessment (SQA) Module measuring Skewness and SNR."""
    n = len(bvp_signal)
    if n < 3:
        return 0.0, 0.0
    mean = np.mean(bvp_signal)
    std = np.std(bvp_signal) + 1e-6
    skewness = (sum((bvp_signal - mean) ** 3) / n) / (std ** 3)
    
    fft_vals = np.abs(np.fft.rfft(bvp_signal))
    freqs = np.fft.rfftfreq(n, d=1.0/fps)
    
    band_mask = (freqs >= 0.7) & (freqs <= 4.0)
    in_band_power = np.sum(fft_vals[band_mask] ** 2)
    total_power = np.sum(fft_vals ** 2) + 1e-6
    snr = in_band_power / total_power
    
    is_clean = (abs(skewness) < 2.0) and (snr > 0.4)
    return float(snr), 1.0 if is_clean else 0.0

def process_single_video_thread(task_info):
    """
    Worker function executed by individual threads.
    Thread-safe processing offloading structural tracking arrays onto the GPU backend.
    """
    video_input_path, output_npy_path, model_path, roi_landmarks_dict = task_info
    
    if os.path.exists(output_npy_path):
        return True # Skip pre-computed runs
        
    # OPTIMIZATION 1: Explicitly force the underlying MediaPipe models onto the GPU backend hardware
    base_options = python.BaseOptions(
        model_asset_path=model_path,
        delegate=python.BaseOptions.Delegate.GPU
    )
    options = vision.FaceLandmarkerOptions(
        base_options=base_options, running_mode=vision.RunningMode.IMAGE, num_faces=1
    )
    
    try:
        local_detector = vision.FaceLandmarker.create_from_options(options)
    except Exception:
        # Graceful runtime fallback to CPU delegate if system GPU driver hooks are unlinked or busy
        base_options = python.BaseOptions(model_asset_path=model_path, delegate=python.BaseOptions.Delegate.CPU)
        options = vision.FaceLandmarkerOptions(base_options=base_options, running_mode=vision.RunningMode.IMAGE, num_faces=1)
        local_detector = vision.FaceLandmarker.create_from_options(options)
    
    cap = cv2.VideoCapture(video_input_path)
    
    # Calculate tracking baseline metric scale boundaries
    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    if orig_fps == 0 or np.isnan(orig_fps): 
        orig_fps = 30.0
    
    # Target effective calculation frame rate based on temporal sub-sampling steps
    effective_fps = orig_fps / 2.0 
        
    roi_traces = {zone: [] for zone in roi_landmarks_dict.keys()}
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
            
        # OPTIMIZATION 2: Temporal Subsampling - skip alternating frames to cut computational tasks by 50%
        if frame_idx % 2 != 0:
            frame_idx += 1
            continue
            
        # OPTIMIZATION 3: Geometric Downscaling - scale frames down to 480p to lower image processing overhead
        frame = cv2.resize(frame, (640, 480), interpolation=cv2.INTER_LINEAR)
        h, w, _ = frame.shape
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        
        detection_result = local_detector.detect(mp_image)
        
        if detection_result.face_landmarks:
            landmarks = detection_result.face_landmarks[0]
            affine_mat = compute_affine_stabilization(landmarks, w, h)
            
            for zone_name, indices in roi_landmarks_dict.items():
                # 1. Create spatial mask for the specific ROI zone on the raw frame
                mask = np.zeros((h, w), dtype=np.uint8)
                pts = np.array([(int(landmarks[idx].x * w), int(landmarks[idx].y * h)) for idx in indices])
                cv2.fillConvexPoly(mask, pts, 255)
                
                # 2. Extract ROI and warp it into a stable coordinate frame
                roi_crop = cv2.bitwise_and(frame, frame, mask=mask)
                warped_roi = cv2.warpAffine(roi_crop, affine_mat, (128, 128))
                
                # 3. Create a matching RGB version of the stabilized crop for mean tracking
                roi_crop_rgb = cv2.bitwise_and(frame_rgb, frame_rgb, mask=mask)
                warped_roi_rgb = cv2.warpAffine(roi_crop_rgb, affine_mat, (128, 128))
                
                # 4. Perform Otsu skin segmentation inside the stabilized plane
                # CRITICAL FIX: Use the updated segmentation routine that safely ignores pure black boundaries
                skin_mask = segment_skin_otsu(warped_roi)
                
                if skin_mask is not None and np.sum(skin_mask) > 0:
                    mean_color = cv2.mean(warped_roi_rgb, mask=skin_mask)[:3]
                else:
                    # Clean boundary fallback logic if segmentation yields an empty matrix array footprint
                    stable_mask = ((warped_roi[..., 0] > 0) & (warped_roi[..., 1] > 0) & (warped_roi[..., 2] > 0)).astype(np.uint8) * 255
                    mean_color = cv2.mean(warped_roi_rgb, mask=stable_mask)[:3]
                    
                roi_traces[zone_name].append(mean_color)
        else:
            for zone_name in roi_landmarks_dict.keys():
                if len(roi_traces[zone_name]) > 0:
                    roi_traces[zone_name].append(roi_traces[zone_name][-1])
                else:
                    roi_traces[zone_name].append([0.0, 0.0, 0.0])
                    
        frame_idx += 1
                    
    cap.release()
    local_detector.close()
    
    processed_roi_signals = []
    quality_metrics = []
    
    for zone_name in roi_landmarks_dict.keys():
        trace_arr = np.array(roi_traces[zone_name])
        # Safe structural fallback array padding for highly clipped sequences
        if len(trace_arr) < 10:
            trace_arr = np.zeros((300, 3)) # Adjusted to target window step length base
            
        pos_signal = apply_pos_rppg(trace_arr)
        filtered_signal = butter_bandpass_filter(pos_signal, fps=effective_fps)
        snr_val, clean_flag = assess_signal_quality(filtered_signal, fps=effective_fps)
        
        processed_roi_signals.append(filtered_signal)
        quality_metrics.append([snr_val, clean_flag])
        
    payload_signals = np.stack(processed_roi_signals)
    payload_metrics = np.array(quality_metrics)
    
    storage_dict = {"signals": payload_signals, "metrics": payload_metrics}
    np.save(output_npy_path, storage_dict, allow_pickle=True)
    return True

# --- EXECUTABLE SCHEDULER (PATH-STABILIZED FOR WINDOWS) ---
master_df = pd.read_csv(CSV_FILE_PATH)

unique_pids = master_df['patient_id'].unique()
development_df = master_df.reset_index(drop=True)

tasks = []
for index, row in development_df.iterrows():
    video_filename = str(row['video']).replace('\\', '/').split('/')[-1]
    video_input_path = os.path.normpath(os.path.join(BASE_DATASET_DIR, 'video', video_filename))
    
    output_npy_basename = f"{os.path.splitext(video_filename)[0]}.npy"
    output_npy_path = os.path.normpath(os.path.join(OUTPUT_SIGNAL_DIR, output_npy_basename))
    
    if os.path.exists(video_input_path):
        tasks.append((video_input_path, output_npy_path, MODEL_PATH, ROI_LANDMARKS))

# NOTE: Cap thread density when utilizing intense parallel GPU context blocks to keep host VRAM stable
max_workers = min(16, max(1, os.cpu_count() - 2)) 
print(f"🎬 Initializing path-stabilized ThreadPool for {len(tasks)} videos using {max_workers} threads...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_single_video_thread, task): task for task in tasks}
    
    # OPTIMIZATION 4: Using robust pure-text tracking loops to eradicate notebook environment exceptions
    with text_tqdm(total=len(futures), desc="⚡ Threaded Extraction") as pbar:
        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception as exc:
                print(f"⚠️ Worker thread raised an internal exception: {exc}")
            pbar.update(1)

print("🎉 Cell 4 Extractor Stable: Data successfully cached via thread pools.")

🎬 Initializing path-stabilized ThreadPool for 3600 videos using 16 threads...


⚡ Threaded Extraction: 100%|██████████| 3600/3600 [25:29:54<00:00, 25.50s/it]   

🎉 Cell 4 Extractor Stable: Data successfully cached via thread pools.


In [5]:
# CELL 5: OVERLAPPING SLIDING WINDOW PYTORCH DATASET GENERATOR (LAZY LOADING CONTEXT)
# =========================================================================================
class ProductionSlidingWindowDataset(Dataset):
    def __init__(self, metadata_df, signal_dir, window_sec=10.0, overlap_pct=0.5, fps=15.0):
        self.signal_dir = signal_dir
        self.window_size = int(window_sec * fps)  # 10s * 15fps = 150 frames
        self.stride = int(self.window_size * (1.0 - overlap_pct))
        
        self.index_map = [] 
        missing_count = 0
        
        print("📦 Indexing patient signal window positions safely...")
        available_columns = metadata_df.columns.tolist()
        expected_targets = ['pulse', 'upper_ap', 'lower_ap', 'saturation', 
                            'hemoglobin', 'stress', 'glycated_hemoglobin', 
                            'cholesterol', 'temperature']
        
        target_keys = [k for k in expected_targets if k in available_columns]
        
        for idx, row in metadata_df.iterrows():
            video_filename = os.path.basename(str(row['video']).replace('\\', '/'))
            npy_basename = f"{os.path.splitext(video_filename)[0]}.npy"
            npy_full_path = os.path.normpath(os.path.join(self.signal_dir, npy_basename))
            
            if not os.path.exists(npy_full_path):
                missing_count += 1
                continue
                
            try:
                data_payload = np.load(npy_full_path, allow_pickle=True).item()
                total_frames = data_payload["signals"].shape[1]
            except Exception:
                continue 
            
            if total_frames < self.window_size:
                continue
                
            start_frame = 0
            while start_frame + self.window_size <= total_frames:
                self.index_map.append({
                    "npy_path": npy_full_path,
                    "start_frame": start_frame,
                    "targets": {k: float(row[k]) for k in target_keys}
                })
                start_frame += self.stride

        print(f"📊 Dataset successfully mapped: {len(self.index_map)} total windows indexed.")
        if missing_count > 0:
            print(f"⚠️ Warning: Could not find matching cached .npy arrays for {missing_count} rows.")

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        meta = self.index_map[idx]
        data_payload = np.load(meta["npy_path"], allow_pickle=True).item()
        signals = data_payload["signals"]
        metrics = data_payload["metrics"]
        
        start = meta["start_frame"]
        end = start + self.window_size
        
        window_slice = signals[:, start:end]
        
        window_mean = np.mean(window_slice, axis=1, keepdims=True)
        window_std = np.std(window_slice, axis=1, keepdims=True)
        
        # NaN-SHIELD: Block division by zero on dead channels/flatlines (e.g. padding regions)
        std_mask = window_std > 1e-5
        normalized_window = np.zeros_like(window_slice)
        normalized_window[std_mask[:, 0]] = (window_slice[std_mask[:, 0]] - window_mean[std_mask[:, 0]]) / (window_std[std_mask[:, 0]] + 1e-6)
        
        x_signal = torch.tensor(normalized_window, dtype=torch.float32)
        x_sqa = torch.tensor(metrics, dtype=torch.float32)
        targets = {key: torch.tensor(val, dtype=torch.float32) for key, val in meta["targets"].items()}
        
        return x_signal, x_sqa, targets

print("✅ Cell 5 Complete: Dataset compiled with explicit NaN protection and 150-frame enforcement.")

✅ Cell 5 Complete: Dataset compiled with explicit NaN protection and 150-frame enforcement.


In [6]:
# CELL 6: MULTI-TASK SCNN MODEL DEFINITION (PYTORCH)
# =========================================================================================
class MultiTaskSCNNNet(nn.Module):
    def __init__(self, num_rois=8, num_tasks=9):
        super(MultiTaskSCNNNet, self).__init__()
        
        # Shared SCNN Encoder Component (Calibrated strictly for input shape [B, 1, 8, 150])
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3, 15), stride=1, padding=(1, 7)),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)), 
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 9), stride=1, padding=(1, 4)),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)), 
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 5), stride=1, padding=(1, 2)),
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=(1, 2)), 
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(8, 3), stride=1, padding=(0, 1)),
            nn.BatchNorm2d(128),
            nn.ELU(),
            nn.AdaptiveAvgPool2d((1, 1)) 
        )
        
        fused_dimension = 128 + (num_rois * 2)
        
        self.shared_dense = nn.Sequential(
            nn.Linear(fused_dimension, 256),
            nn.ELU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 128),
            nn.ELU(),
            nn.Dropout(p=0.2)
        )
        
        # Multi-Task Prediction Heads
        self.head_pulse               = nn.Linear(128, 1)
        self.head_upper_ap            = nn.Linear(128, 1)
        self.head_lower_ap            = nn.Linear(128, 1)
        self.head_saturation          = nn.Linear(128, 1)
        self.head_hemoglobin          = nn.Linear(128, 1)
        self.head_stress              = nn.Linear(128, 1)
        self.head_glycated_hemoglobin = nn.Linear(128, 1)
        self.head_cholesterol         = nn.Linear(128, 1)
        self.head_temperature         = nn.Linear(128, 1)
        
        # FIXED: Expanded log-variance output layer to yield 9 independent uncertainty allocations
        self.head_confidence = nn.Sequential(
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, num_tasks)
        )

    def forward(self, signal_window, sqa_metrics):
        x = signal_window.unsqueeze(1) # Conversion from [B, 8, 150] to 4D tensor [B, 1, 8, 150]
        visual_features = self.encoder(x).view(x.size(0), -1) 
        sqa_features = sqa_metrics.view(sqa_metrics.size(0), -1)
        
        fused_vector = torch.cat((visual_features, sqa_features), dim=1)
        shared_out = self.shared_dense(fused_vector)
        
        # Pull the task-decoupled multi-uncertainty matrix array
        log_vars = self.head_confidence(shared_out) # Shape: [B, 9]
        
        predictions = {
            'pulse':      self.head_pulse(shared_out).squeeze(-1),
            'upper_ap':   self.head_upper_ap(shared_out).squeeze(-1),
            'lower_ap':   self.head_lower_ap(shared_out).squeeze(-1),
            'saturation': self.head_saturation(shared_out).squeeze(-1),
            'hemoglobin': self.head_hemoglobin(shared_out).squeeze(-1),
            'stress':     self.head_stress(shared_out).squeeze(-1),
            'glycated_hemoglobin': self.head_glycated_hemoglobin(shared_out).squeeze(-1),
            'cholesterol':        self.head_cholesterol(shared_out).squeeze(-1),
            'temperature':        self.head_temperature(shared_out).squeeze(-1),
            'log_vars':           log_vars, # Dict bound mapping multi-uncertainty tokens cleanly
        }
        return predictions

print("✅ Cell 6 Complete: Model architecture bound with independent multi-task variance estimators.")

✅ Cell 6 Complete: Model architecture bound with independent multi-task variance estimators.


In [7]:
# CELL 7: LEAK-FREE SPLITTING, STABILIZED LOSS OPTIMIZATION & TRAINING ENGINE
# =========================================================================================
# Pipeline Stage: Strategic Evaluation Grouping, Homoscedastic Multi-Task Training Loop
# Algorithms: Patient-Stratified Data Splitting, AdamW Optimizer, Scikit-Learn R² Validation
# =========================================================================================
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

TARGET_KEYS = ['pulse', 'upper_ap', 'lower_ap', 'saturation', 
               'hemoglobin', 'stress', 'glycated_hemoglobin', 
               'cholesterol', 'temperature']

# STEP 1: Split cohort using patient boundaries first to guarantee leak prevention
np.random.seed(42)
shuffled_pids = np.copy(unique_pids)
np.random.shuffle(shuffled_pids)

split_idx = int(len(shuffled_pids) * 0.8)
train_pids, val_pids = shuffled_pids[:split_idx], shuffled_pids[split_idx:]

train_df_split = development_df[development_df['patient_id'].isin(train_pids)].copy().reset_index(drop=True)
val_df_split = development_df[development_df['patient_id'].isin(val_pids)].copy().reset_index(drop=True)

# STEP 2: Leak-Free scaling applied safely POST-split
scaler = MinMaxScaler(feature_range=(0.1, 0.9))
train_df_split[TARGET_KEYS] = scaler.fit_transform(train_df_split[TARGET_KEYS])
val_df_split[TARGET_KEYS] = scaler.transform(val_df_split[TARGET_KEYS])

train_dataset = ProductionSlidingWindowDataset(train_df_split, OUTPUT_SIGNAL_DIR, window_sec=10.0, overlap_pct=0.5, fps=15.0)
val_dataset = ProductionSlidingWindowDataset(val_df_split, OUTPUT_SIGNAL_DIR, window_sec=10.0, overlap_pct=0.5, fps=15.0)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = MultiTaskSCNNNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

TASK_WEIGHTS = {
    'pulse': 0.1, 'upper_ap': 0.05, 'lower_ap': 0.05, 'saturation': 1.0,
    'hemoglobin': 0.5, 'stress': 0.5, 'glycated_hemoglobin': 1.0, 
    'cholesterol': 0.5, 'temperature': 1.0
}

TOTAL_EPOCHS = 15
print(f"🏋️ Starting Multi-Task Training Loop for {TOTAL_EPOCHS} Epochs on target: {DEVICE}...\n")

for epoch in range(1, TOTAL_EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    
    for signals, sqa_m, targets in train_loader:
        signals, sqa_m = signals.to(DEVICE), sqa_m.to(DEVICE)
        optimizer.zero_grad()
        
        preds = model(signals, sqa_m)
        total_loss = 0.0
        
        # NUMERICAL STABILITY Clamping across the whole uncertainty distribution slice
        log_vars = torch.clamp(preds['log_vars'], min=-5.0, max=5.0)
        
        for idx, task_name in enumerate(TARGET_KEYS):
            task_mse = F.mse_loss(preds[task_name], targets[task_name].to(DEVICE), reduction='none')
            s_task = log_vars[:, idx] # Extract target uncertainty for this specific physiological slice
            
            # FIXED DECOUPLED HETEROSCEDASTIC UNCERTAINTY EQUATION:
            # Balancing penalty calculations element-wise across the batch cleanly
            weighted_task_loss = torch.mean(torch.exp(-s_task) * (TASK_WEIGHTS[task_name] * task_mse) + TASK_WEIGHTS[task_name] * s_task)
            total_loss += weighted_task_loss
            
        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        running_train_loss += total_loss.item()

    # --- VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0
    val_metrics_accum = {key: {'true': [], 'pred': []} for key in TARGET_KEYS}
    inferred_variance_scores = {key: [] for key in TARGET_KEYS}
    
    with torch.no_grad():
        for signals, sqa_m, targets in val_loader:
            signals, sqa_m = signals.to(DEVICE), sqa_m.to(DEVICE)
            preds = model(signals, sqa_m)
            
            log_vars = torch.clamp(preds['log_vars'], min=-5.0, max=5.0)
            
            total_loss = 0.0
            for idx, task_name in enumerate(TARGET_KEYS):
                task_mse = F.mse_loss(preds[task_name], targets[task_name].to(DEVICE), reduction='none')
                s_task = log_vars[:, idx]
                
                weighted_task_loss = torch.mean(torch.exp(-s_task) * (TASK_WEIGHTS[task_name] * task_mse) + TASK_WEIGHTS[task_name] * s_task)
                total_loss += weighted_task_loss
                
                # Cache variance measurements decoupled by metric index positioning
                inferred_variance_scores[task_name].extend(torch.exp(s_task).cpu().numpy().tolist())
            
            for task_name in TARGET_KEYS:
                val_metrics_accum[task_name]['true'].extend(targets[task_name].cpu().numpy().tolist())
                val_metrics_accum[task_name]['pred'].extend(preds[task_name].cpu().numpy().tolist())
                
            running_val_loss += total_loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    avg_val_loss = running_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    # Calculate global mean variance representation for performance tracking logs
    global_mean_var = np.mean([np.mean(inferred_variance_scores[k]) for k in TARGET_KEYS])
    print(f"📈 Epoch [{epoch:02d}/{TOTAL_EPOCHS:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Avg Global Var: {global_mean_var:.4f}")

    n_val_samples = len(val_metrics_accum[TARGET_KEYS[0]]['true'])
    if n_val_samples > 0:
        scaled_true_matrix = np.zeros((n_val_samples, len(TARGET_KEYS)))
        scaled_pred_matrix = np.zeros((n_val_samples, len(TARGET_KEYS)))
        
        for idx, task_name in enumerate(TARGET_KEYS):
            scaled_true_matrix[:, idx] = val_metrics_accum[task_name]['true']
            scaled_pred_matrix[:, idx] = val_metrics_accum[task_name]['pred']
            
        unscaled_true = scaler.inverse_transform(scaled_true_matrix)
        unscaled_pred = scaler.inverse_transform(scaled_pred_matrix)
        
        print("   Physiological Unit Validation Performance:")
        for idx, task_name in enumerate(TARGET_KEYS):
            task_mae = mean_absolute_error(unscaled_true[:, idx], unscaled_pred[:, idx])
            task_r2 = r2_score(unscaled_true[:, idx], unscaled_pred[:, idx])
            print(f"      ↳ {task_name:<20} : MAE = {task_mae:.4f} | R² = {task_r2:.4f} (Avg Var: {np.mean(inferred_variance_scores[task_name]):.4f})")
            
    print("-" * 105)

torch.save(model.state_dict(), os.path.join(BASE_PROJECT_DIR, "production_scnn_approach3.pth"))
print("💾 Model weight matrices successfully cached to project root.")

📦 Indexing patient signal window positions safely...
📊 Dataset successfully mapped: 91200 total windows indexed.
📦 Indexing patient signal window positions safely...
📊 Dataset successfully mapped: 22791 total windows indexed.
🏋️ Starting Multi-Task Training Loop for 15 Epochs on target: cuda...

📈 Epoch [01/15] | Train Loss: -17.0224 | Val Loss: -18.4122 | Avg Global Var: 0.0149
   Physiological Unit Validation Performance:
      ↳ pulse                : MAE = 14.7543 | R² = -0.0171 (Avg Var: 0.0191)
      ↳ upper_ap             : MAE = 14.2040 | R² = -0.0070 (Avg Var: 0.0126)
      ↳ lower_ap             : MAE = 7.1405 | R² = -0.0150 (Avg Var: 0.0182)
      ↳ saturation           : MAE = 1.1234 | R² = -0.1065 (Avg Var: 0.0067)
      ↳ hemoglobin           : MAE = 1.2361 | R² = -0.1020 (Avg Var: 0.0198)
      ↳ stress               : MAE = 1.0765 | R² = -0.0308 (Avg Var: 0.0342)
      ↳ glycated_hemoglobin  : MAE = 0.3670 | R² = 0.0048 (Avg Var: 0.0067)
      ↳ cholesterol          : M

In [9]:
# CELL 7.1: REVISED BULLETPROOF MULTI-TASK TRAINING ENGINE (NO-CHEAT LOSS)
# =========================================================================================
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

TARGET_KEYS = ['pulse', 'upper_ap', 'lower_ap', 'saturation', 
               'hemoglobin', 'stress', 'glycated_hemoglobin', 
               'cholesterol', 'temperature']

# STEP 1: Split cohort cleanly
np.random.seed(42)
shuffled_pids = np.copy(unique_pids)
np.random.shuffle(shuffled_pids)

split_idx = int(len(shuffled_pids) * 0.8)
train_pids, val_pids = shuffled_pids[:split_idx], shuffled_pids[split_idx:]

train_df_split = development_df[development_df['patient_id'].isin(train_pids)].copy().reset_index(drop=True)
val_df_split = development_df[development_df['patient_id'].isin(val_pids)].copy().reset_index(drop=True)

# STEP 2: Safe scaling
scaler = MinMaxScaler(feature_range=(0.1, 0.9))
train_df_split[TARGET_KEYS] = scaler.fit_transform(train_df_split[TARGET_KEYS])
val_df_split[TARGET_KEYS] = scaler.transform(val_df_split[TARGET_KEYS])

train_dataset = ProductionSlidingWindowDataset(train_df_split, OUTPUT_SIGNAL_DIR, window_sec=10.0, overlap_pct=0.5, fps=15.0)
val_dataset = ProductionSlidingWindowDataset(val_df_split, OUTPUT_SIGNAL_DIR, window_sec=10.0, overlap_pct=0.5, fps=15.0)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = MultiTaskSCNNNet().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.05) # Lower LR + higher decay to combat mean prediction
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Adjusted Task Weights to prioritize highly dynamic vitals
TASK_WEIGHTS = {
    'pulse': 2.0, 'upper_ap': 1.5, 'lower_ap': 1.5, 'saturation': 1.0,
    'hemoglobin': 1.0, 'stress': 1.0, 'glycated_hemoglobin': 1.0, 
    'cholesterol': 1.0, 'temperature': 1.0
}

TOTAL_EPOCHS = 20
print(f"🏋️ Relaunching Cleaned Multi-Task Training on: {DEVICE}...\n")

for epoch in range(1, TOTAL_EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    
    for signals, sqa_m, targets in train_loader:
        signals, sqa_m = signals.to(DEVICE), sqa_m.to(DEVICE)
        optimizer.zero_grad()
        
        preds = model(signals, sqa_m)
        total_loss = 0.0
        
        for task_name in TARGET_KEYS:
            # Replaced MSE with Smooth L1 Loss to prevent steep outlier distortions
            task_loss = F.smooth_l1_loss(preds[task_name], targets[task_name].to(DEVICE), beta=0.1)
            total_loss += TASK_WEIGHTS[task_name] * task_loss
            
        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Tighter gradient clipping
        optimizer.step()
        running_train_loss += total_loss.item()

    # --- VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0
    val_metrics_accum = {key: {'true': [], 'pred': []} for key in TARGET_KEYS}
    
    with torch.no_grad():
        for signals, sqa_m, targets in val_loader:
            signals, sqa_m = signals.to(DEVICE), sqa_m.to(DEVICE)
            preds = model(signals, sqa_m)
            
            total_loss = 0.0
            for task_name in TARGET_KEYS:
                task_loss = F.smooth_l1_loss(preds[task_name], targets[task_name].to(DEVICE), beta=0.1)
                total_loss += TASK_WEIGHTS[task_name] * task_loss
            
            for task_name in TARGET_KEYS:
                val_metrics_accum[task_name]['true'].extend(targets[task_name].cpu().numpy().tolist())
                val_metrics_accum[task_name]['pred'].extend(preds[task_name].cpu().numpy().tolist())
                
            running_val_loss += total_loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    avg_val_loss = running_val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    
    print(f"📈 Epoch [{epoch:02d}/{TOTAL_EPOCHS:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    n_val_samples = len(val_metrics_accum[TARGET_KEYS[0]]['true'])
    if n_val_samples > 0:
        scaled_true_matrix = np.zeros((n_val_samples, len(TARGET_KEYS)))
        scaled_pred_matrix = np.zeros((n_val_samples, len(TARGET_KEYS)))
        
        for idx, task_name in enumerate(TARGET_KEYS):
            scaled_true_matrix[:, idx] = val_metrics_accum[task_name]['true']
            scaled_pred_matrix[:, idx] = val_metrics_accum[task_name]['pred']
            
        unscaled_true = scaler.inverse_transform(scaled_true_matrix)
        unscaled_pred = scaler.inverse_transform(scaled_pred_matrix)
        
        print("   Physiological Unit Validation Performance:")
        for idx, task_name in enumerate(TARGET_KEYS):
            task_mae = mean_absolute_error(unscaled_true[:, idx], unscaled_pred[:, idx])
            task_r2 = r2_score(unscaled_true[:, idx], unscaled_pred[:, idx])
            print(f"      ↳ {task_name:<20} : MAE = {task_mae:.4f} | R² = {task_r2:.4f}")
            
    print("-" * 75)

torch.save(model.state_dict(), os.path.join(BASE_PROJECT_DIR, "production_scnn2_approach3.pth"))
print("💾 Model weight matrices successfully cached to project root.")

📦 Indexing patient signal window positions safely...
📊 Dataset successfully mapped: 91200 total windows indexed.
📦 Indexing patient signal window positions safely...
📊 Dataset successfully mapped: 22791 total windows indexed.
🏋️ Relaunching Cleaned Multi-Task Training on: cuda...

📈 Epoch [01/20] | Train Loss: 0.6572 | Val Loss: 0.5656
   Physiological Unit Validation Performance:
      ↳ pulse                : MAE = 15.0012 | R² = -0.0763
      ↳ upper_ap             : MAE = 13.6573 | R² = -0.0043
      ↳ lower_ap             : MAE = 7.0874 | R² = 0.0071
      ↳ saturation           : MAE = 0.9328 | R² = -0.0088
      ↳ hemoglobin           : MAE = 1.1787 | R² = -0.0511
      ↳ stress               : MAE = 1.0757 | R² = -0.0325
      ↳ glycated_hemoglobin  : MAE = 0.3776 | R² = -0.0260
      ↳ cholesterol          : MAE = 0.6005 | R² = -0.0131
      ↳ temperature          : MAE = 0.0951 | R² = -0.0069
---------------------------------------------------------------------------
📈 Epoch 

In [8]:
# CELL 8: ALIGNED PRODUCTION INFERENCE ENGINE (FIXED SHAPE & KEY MAPPING)
# =========================================================================================
def predict_patient_vitals(npy_signal_path, eval_model, device, scaler, target_keys):
    """
    Unified Production Validation Engine calibrated for 15 FPS sub-sampled matrix arrays.
    Validates, cuts, and extracts results using independent task-level confidence bounds.
    """
    if not os.path.exists(npy_signal_path):
        raise FileNotFoundError(f"Missing precomputed matrix: {npy_signal_path}")
        
    data_payload = np.load(npy_signal_path, allow_pickle=True).item()
    signals = data_payload["signals"]   # Expected original shape: [8, Total_Frames]
    metrics = data_payload["metrics"]   # Expected shape: [8, 2]
    
    # ENFORCED ALIGNMENT: 150-frame check guarantees parity with spatial feature map limits
    if signals.shape[1] < 150:
        raise ValueError(f"Signal matrix window length ({signals.shape[1]}) is lower than the mandatory 150 frames.")
        
    sliced_signals = signals[:, :150]
    
    # Dynamic slice normalization with safe division shield
    window_mean = np.mean(sliced_signals, axis=1, keepdims=True)
    window_std = np.std(sliced_signals, axis=1, keepdims=True)
    
    std_mask = window_std > 1e-5
    normalized_window = np.zeros_like(sliced_signals)
    normalized_window[std_mask[:, 0]] = (sliced_signals[std_mask[:, 0]] - window_mean[std_mask[:, 0]]) / (window_std[std_mask[:, 0]] + 1e-6)
    
    # 4D PARITY UNSQUEEZE PIPELINE: [8, 150] -> [1, 8, 150] -> model will perform .unsqueeze(1) -> [1, 1, 8, 150]
    x_signal = torch.tensor(normalized_window, dtype=torch.float32).unsqueeze(0).to(device)
    x_sqa = torch.tensor(metrics, dtype=torch.float32).unsqueeze(0).to(device)
    
    eval_model.eval()
    with torch.no_grad():
        preds = eval_model(x_signal, x_sqa)
        
    scaled_output_vector = np.zeros((1, len(target_keys)))
    for idx, task_name in enumerate(target_keys):
        scaled_output_vector[0, idx] = preds[task_name].cpu().numpy()[0]
        
    unscaled_predictions = scaler.inverse_transform(scaled_output_vector)[0]
    
    # Extract independent log variance vector outputs safely
    log_vars_output = preds['log_vars'].cpu().numpy()[0] # Shape: [9]
    # FIXED: Bound limits adjusted to match Cell 7 (-5.0, 5.0) for absolute mathematical parity
    clamped_log_vars = np.clip(log_vars_output, -5.0, 5.0)
    
    results = {}
    for idx, task_name in enumerate(target_keys):
        # Map task-specific metric variance to bounded individual confidence values
        inferred_task_var = np.exp(clamped_log_vars[idx])
        task_confidence = 1.0 / (1.0 + inferred_task_var)
        
        results[task_name] = float(unscaled_predictions[idx])
        results[f'{task_name}_confidence'] = float(task_confidence)
        
    # Return global confidence average as basic benchmark overview tracking metric token
    results['global_model_confidence'] = float(1.0 / (1.0 + np.exp(np.mean(clamped_log_vars))))
    
    return results

print("✅ Cell 8 Complete: Production inference accurately bound to 150-frame inputs with task-decoupled metrics.")

✅ Cell 8 Complete: Production inference accurately bound to 150-frame inputs with task-decoupled metrics.
